# 🫁 Huấn luyện, Tối ưu hóa AUROC & So sánh Mô hình Lai `HybridCNNViT` (`Base` vs. `Large`)

Notebook này được thiết kế để chạy trực tiếp trên **Colab Extension trong Antigravity** (kết nối GPU A100/L4), bao gồm:
1. **Khởi tạo môi trường, Mount Google Drive & Giải nén dữ liệu** (`126,884` ảnh X-quang ngực từ `NIH ChestX-ray14` + `VinDr-CXR`).
2. **Lịch sử thực nghiệm cũ (Baseline `Large` v1 — `Test AUROC = 0.8759`)** & Phân tích 6 nguyên nhân gây trần AUROC.
3. **Kiểm toán độ cân bằng dữ liệu (`Dataset & Imbalance Audit`) & Benchmark kiến trúc (`Base` vs. `Large`)**.
4. **Huấn luyện mới bản `Base` (`ConvNeXtV2-Base + SwinV2-Base`, `125.26M` params, 384×384)** từ đầu bằng pipeline v2 đã tối ưu.
5. **Huấn luyện mới bản `Large` (`ConvNeXtV2-Large + SwinV2-Large`, `279.12M` params, 384×384)** từ đầu bằng cùng pipeline v2 để đảm bảo so sánh công bằng (`Fair Comparison`) và đạt AUROC tối đa (kèm tùy chọn Fine-tune nhanh SOTA v2.1).
6. **Đánh giá Test Set (4-View Multi-Resolution TTA) & So sánh đối đầu (`Head-to-Head Evaluation`)**: `Base` vs. `Large` vs. `Base + Large Ensemble` với khoảng tin cậy **Bootstrap 95% CI**.


In [ ]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

if not os.path.exists('/content/CheXNet'):
    !git clone -b dev https://github.com/lecuong2512/CheXNet /content/CheXNet
else:
    %cd /content/CheXNet
    !git fetch origin dev && git checkout dev && git pull origin dev
%cd /content/CheXNet


Cloning into 'CheXNet'...
remote: Enumerating objects: 569, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 569 (delta 82), reused 87 (delta 44), pack-reused 430 (from 1)
Receiving objects: 100% (569/569), 81.31 MiB | 48.26 MiB/s, done.
Resolving deltas: 100% (224/224), done.


In [ ]:
import os
import shutil
import zipfile
import time
import glob
import re
from pathlib import Path

# --- HÀM HỖ TRỢ SẮP XẾP TỰ NHIÊN (Để data_2 đứng trước data_10) ---
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', s)]

# --- CẤU HÌNH ---
SOURCE_DIR = "/content/drive/MyDrive/Database_512"
TARGET_DIR = "/content/CheXNet/Database"

# Tạo thư mục đích
os.makedirs(TARGET_DIR, exist_ok=True)

print("🚀 Đang quét tìm các tệp .zip trong thư mục nguồn...")

# 1. TỰ ĐỘNG TÌM KIẾM FILE ZIP
# Lấy tất cả file có đuôi .zip
search_pattern = os.path.join(SOURCE_DIR, "*.zip")
zip_files = glob.glob(search_pattern)

# Sắp xếp danh sách file cho gọn gàng
zip_files.sort(key=natural_sort_key)

total_files = len(zip_files)

if total_files == 0:
    print(f"⚠️  Không tìm thấy tệp .zip nào trong: {SOURCE_DIR}")
else:
    print(f"✅ Đã tìm thấy {total_files} tệp zip.")
    print("-" * 50)

    # 2. VÒNG LẶP XỬ LÝ TỪNG FILE
    for i, source_path in enumerate(zip_files, 1):
        file_name = os.path.basename(source_path) # Lấy tên file (ví dụ: data_1.zip)
        temp_file = os.path.join("/content", file_name)

        print(f"📦 [{i}/{total_files}] Đang xử lý: {file_name}")

        # --- A. COPY TỪ DRIVE VÀO TEMP ---
        file_size = os.path.getsize(source_path)
        print(f"   📥 Đang sao chép... ({file_size / 1024 / 1024:.1f} MB)")
        start_time = time.time()

        BUFFER_SIZE = 1024 * 1024  # 1MB buffer
        copied = 0

        try:
            with open(source_path, 'rb') as src, open(temp_file, 'wb') as dst:
                while True:
                    buffer = src.read(BUFFER_SIZE)
                    if not buffer:
                        break
                    dst.write(buffer)
                    copied += len(buffer)

                    # Hiển thị tiến độ copy
                    duration = time.time() - start_time
                    if duration > 0:
                        speed = copied / 1024 / 1024 / duration
                        percent = copied * 100 / file_size
                        print(f"\r   ⏳ Copy: {percent:5.1f}% | {speed:6.2f} MB/s", end='', flush=True)

            print() # Xuống dòng sau khi copy xong

            # --- B. GIẢI NÉN ---
            print(f"   📂 Đang giải nén vào đích...")
            unzip_start = time.time()

            with zipfile.ZipFile(temp_file, 'r') as zip_ref:
                file_list = zip_ref.namelist()
                total_items = len(file_list)

                for idx, file in enumerate(file_list, 1):
                    zip_ref.extract(file, TARGET_DIR)
                    # Chỉ cập nhật hiển thị mỗi 50 file để đỡ lag terminal nếu quá nhiều file nhỏ
                    if idx % 50 == 0 or idx == total_items:
                        percent = idx * 100 / total_items
                        print(f"\r   🔨 Unzip: {percent:5.1f}% ({idx}/{total_items} files)", end='')

            unzip_duration = time.time() - unzip_start
            print(f"\n   ✅ Giải nén xong trong {unzip_duration:.1f}s")

            # --- C. DỌN DẸP ---
            os.remove(temp_file)
            print(f"   🗑️  Đã xóa file tạm.")
            print("-" * 50)

        except Exception as e:
            print(f"\n❌ LỖI khi xử lý file {file_name}: {e}")
            continue

print(f"\n🎉 HOÀN TẤT! Đã xử lý xong {total_files} file.")

🚀 Đang quét tìm các tệp .zip trong thư mục nguồn...
✅ Đã tìm thấy 5 tệp zip.
--------------------------------------------------
📦 [1/5] Đang xử lý: data_1.zip
   📥 Đang sao chép... (5122.9 MB)
   ⏳ Copy: 100.0% |  52.03 MB/s
   📂 Đang giải nén vào đích...
   🔨 Unzip: 100.0% (27708/27708 files)
   ✅ Giải nén xong trong 17.2s
   🗑️  Đã xóa file tạm.
--------------------------------------------------
📦 [2/5] Đang xử lý: data_2.zip
   📥 Đang sao chép... (5123.1 MB)
   ⏳ Copy: 100.0% |  54.64 MB/s
   📂 Đang giải nén vào đích...
   🔨 Unzip: 100.0% (27980/27980 files)
   ✅ Giải nén xong trong 13.6s
   🗑️  Đã xóa file tạm.
--------------------------------------------------
📦 [3/5] Đang xử lý: data_3.zip
   📥 Đang sao chép... (5122.9 MB)
   ⏳ Copy: 100.0% |  53.45 MB/s
   📂 Đang giải nén vào đích...
   🔨 Unzip: 100.0% (27314/27314 files)
   ✅ Giải nén xong trong 17.7s
   🗑️  Đã xóa file tạm.
--------------------------------------------------
📦 [4/5] Đang xử lý: data_4.zip
   📥 Đang sao chép... 

In [ ]:
import os, shutil

root_dir = '/content/CheXNet/Database'  # 👉 thư mục gốc
target_dir = '/content/CheXNet/Database'  # nơi gom ảnh ra (có thể đổi nếu muốn)

# Các định dạng ảnh cần gom
exts = ('.jpg', '.jpeg', '.png', '.bmp', '.dicom', '.dcm')

count = 0
for subdir, _, files in os.walk(root_dir):
    for f in files:
        if f.lower().endswith(exts):
            src = os.path.join(subdir, f)
            dst = os.path.join(target_dir, f)

            # Move hoặc Copy
            shutil.move(src, dst)
            count += 1

print(f"✅ Đã di chuyển {count} ảnh ra ngoài {target_dir}")

✅ Đã di chuyển 127120 ảnh ra ngoài /content/CheXNet/Database


---
## 📊 Phần A: Lịch sử Huấn luyện Baseline `Large` (v1) & Chẩn đoán Kỹ thuật

Các ô dưới đây giữ lại lịch sử chạy bản `Large` v1 (`Val AUROC = 0.8657` tại Epoch 13, `Test Mean AUROC = 0.8759`) để làm mốc tham chiếu (`Baseline`).
- **Nguyên nhân suy giảm sau Epoch 12 ở bản v1**:
  1. `AttentionSparsityLoss` cũ phạt `L1 = attention_map.mean()` trên toàn bộ 15 kênh và 75% ảnh NIH, làm `attention_map.mean()` sụp đổ từ `0.171` (Epoch 6) xuống `0.020` (Epoch 12) và `0.010` (Epoch 20).
  2. Lịch trình `CosineAnnealingLR(T_max=47)` quá dài giữ `backbone_lr` ở mức cao (`~8.8e-5`) khi mỗi ảnh VinDr lặp `2.53×/epoch`.
  3. `No Finding` (chiếm 55.88% mẫu dương tính) bị áp `gamma_neg = 4, clip = 0.05` làm giảm gradient của các ca có bệnh.
  4. 4 lớp không có Bounding Box (`Pneumonia`, `Edema`, `Emphysema`, `Hernia`) thiếu tín hiệu định vị yếu (MIL Peak).


In [ ]:
!python3 CheXNet/main.py


Group Project: Hybrid CNN-ViT Multi-Label Chest X-ray Classification
Architecture: ConvNeXtV2 & SwinV2 | Residual Masking

Chọn chế độ:
  1. Train mới từ đầu
  2. Train tiếp từ checkpoint cũ (Resume)
  3. Test / Đánh giá
Lựa chọn [1/2/3]: 1

[MODE] Khởi tạo huấn luyện Mô hình Lai

🏗️ Chọn cặp Backbone:
  1. Base-Base (ConvNeXtV2-Base + SwinV2-Base)  — VRAM 12–16 GB
  2. Large-Large (ConvNeXtV2-Large + SwinV2-Large) — VRAM > 24 GB
Lựa chọn [1/2]: 2

📐 Chọn kích thước ảnh đầu vào:
  1. 256×256 (Tốc độ nhanh, tiết kiệm VRAM)
  2. 384×384 (Độ phân giải cao, khuyến nghị cho VinDr)
Lựa chọn [1/2]: 2

🚀 Tensor Core & VRAM Analysis
GPU: NVIDIA A100-SXM4-80GB
Memory: 79.3 GB
Compute Capability: 8.0
✓ Tensor Cores: ENABLED (Mixed Precision / TF32)
✓ AMP: ENABLED (BF16) — VRAM giảm ~40%, throughput tăng 2-3x

💡 Cấu hình đề xuất cho Hybrid (LARGE):
  • Kích thước ảnh: 384x384
  • Batch Size (per GPU): 56
  ⚠ CẢNH BÁO VRAM: Đã tự động kích hoạt Gradient Checkpointing.


Sử dụng Batch Size đề xuất?

In [ ]:
!python3 CheXNet/main.py


Group Project: Hybrid CNN-ViT Multi-Label Chest X-ray Classification
Architecture: ConvNeXtV2 & SwinV2 | Residual Masking

Chọn chế độ:
  1. Train mới từ đầu
  2. Train tiếp từ checkpoint cũ (Resume)
  3. Test / Đánh giá
Lựa chọn [1/2/3]: 3

[MODE] Kiểm thử / Đánh giá Mô hình Lai
Đường dẫn file model (.pth) [Trainedmodel/hybrid_model.pth]: /content/drive/MyDrive/CheXNet/hybrid_model_sota.pth
Thư mục chứa ảnh [Database/]: /content/CheXNet/Database
File CSV Test [Dataset/test_list.csv]: /content/CheXNet/Dataset/test_list.csv
Thư mục lưu kết quả [Results/]: 

Device: cuda
✅ Checkpoint: epoch=13, stage=2, best_auroc=0.8657
   model_size=large, img_size=384

🏗️ Chọn kích thước model:
  1. Base  (mặc định từ checkpoint:  )
  2. Large (mặc định từ checkpoint: ✓)
Lựa chọn [1/2] (Enter = 2): 

📐 Chọn kích thước ảnh:
  1. 256×256   (mặc định từ checkpoint:  )
  2. 384×384   (mặc định từ checkpoint: ✓)
Lựa chọn [1/2] (Enter = 2): 

🏗️ Đang load model (LARGE, 384px)...

Loading dataset from: /con

In [ ]:
!python /content/CheXNet/Finetune/main_finetune.py


🔬 SOTA FINE-TUNE v2 — 3-Phase Strategy
   Pha 1: Freeze Backbone + Warmup Head
   Pha 2: Unfreeze + MixUp + R-Drop
   Pha 3: SWA + Snapshot Ensemble

📂 Đường dẫn checkpoint nguồn (.pth) [Trainedmodel/hybrid_model.pth]: /content/drive/MyDrive/CheXNet/hybrid_model.pth

✅ Checkpoint: epoch=12, stage=2, AUROC=0.8651
   model_size=large, img_size=384

🔒 Khóa cứng: model_size=large, img_size=384

🚀 Tensor Core & VRAM Analysis
GPU: NVIDIA A100-SXM4-40GB
Memory: 39.5 GB
Compute Capability: 8.0
✓ Tensor Cores: ENABLED (Mixed Precision / TF32)
✓ AMP: ENABLED (BF16) — VRAM giảm ~40%, throughput tăng 2-3x

💡 Cấu hình đề xuất cho Hybrid (LARGE):
  • Kích thước ảnh: 384x384
  • Batch Size (per GPU): 24
  ⚠ CẢNH BÁO VRAM: Đã tự động kích hoạt Gradient Checkpointing.


Sử dụng Batch Size đề xuất? [y/n]: n
Nhập Batch Size thủ công (VD: 8, 16, 32): 32

📂 Cấu hình đường dẫn dữ liệu:
Thư mục chứa ảnh [Database/]: /content/CheXNet/Database
File CSV Train [Dataset/train_list.csv]: /content/CheXNet/Dataset/

In [ ]:
!python /content/CheXNet/reduce_model_size.py \
  --input /content/drive/MyDrive/CheXNet/hybrid_model.pth \
  --output /content/drive/MyDrive/CheXNet/hybrid_model_light.pth

Đang tải checkpoint từ: /content/drive/MyDrive/CheXNet/hybrid_model.pth
Vui lòng đợi vài phút do file khá nặng...

Trích xuất các thành phần cần thiết:
  ✓ Đã giữ lại: epoch
  ✓ Đã giữ lại: state_dict
  ✓ Đã giữ lại: training_state_dict
  ✓ Đã giữ lại: best_auroc
  ✓ Đã giữ lại: stage
  ✓ Đã giữ lại: model_size
  ✓ Đã giữ lại: img_size
  ✓ Đã giữ lại: stage1_dice_met_ever

Các thành phần đã bị xóa để giảm dung lượng:
  🗑️ Đã xóa: optimizer_state_dict
  🗑️ Đã xóa: scheduler_state_dict
  🗑️ Đã xóa: scaler_state_dict
  🗑️ Đã xóa: uncertainty_weights_state_dict

Đang lưu checkpoint mới ra: /content/drive/MyDrive/CheXNet/hybrid_model_light.pth

✅ HOÀN TẤT!
Dung lượng cũ: 4.16 GB
Dung lượng mới: 2.08 GB
Đã giảm được: 2.08 GB


---
## 🔎 Bước 1: Kiểm toán Độ Cân bằng Dữ liệu & Benchmark Phần cứng (`Base` vs. `Large`)

Chạy `compare_models.py --hardware-only` trên máy ảo Colab để kiểm tra GPU, cập nhật code mới nhất từ nhánh `dev`, và xuất bảng kiểm toán 15 lớp bệnh lý cùng thông số kiến trúc `Base` (`125.26M`) vs. `Large` (`279.12M`).


In [ ]:
%cd /content/CheXNet
!git fetch origin dev && git checkout dev && git pull origin dev
!python3 compare_models.py --hardware-only \
  --large-ckpt /content/drive/MyDrive/CheXNet/hybrid_model.pth \
  --save-dir /content/CheXNet/Results/Comparison


---
## 🚀 Bước 2: Huấn luyện Mới Bản `Base` Từ Đầu (`ConvNeXtV2-Base + SwinV2-Base`, 384×384)

Huấn luyện bản **`Base` (`125.26M` params)** từ đầu qua 2 giai đoạn với đầy đủ 6 nâng cấp trong `Model.py` và `TrainModel.py`:
- **Stochastic Depth (`drop_path_rate = 0.2`)** + **Multi-Sample Dropout (`5× MSD`)**.
- **Class-Conditional Hinge Area Sparsity + MIL Spatial Peak Loss** giữ bản đồ nhiệt chú ý sắc nét xuyên suốt Stage 2 và định vị yếu cho cả 4 bệnh không có bbox.
- **Adaptive ASL**: `gamma_neg = 1.0, clip = 0.0` cho `No Finding` + trọng số dương tính mềm $w_c \in [1.0, 2.5]$ cho các bệnh hiếm.
- **4-Group AdamW (`decay` vs. `no_decay`)** + tái cân chỉnh `CosineAnnealingLR` ngắn hạn (`14` epochs tổng cộng: ~4–5 epochs Stage 1 + ~9–10 epochs Stage 2).
- Lưu checkpoint tại `/content/drive/MyDrive/CheXNet/hybrid_base.pth` và ghi log trực tiếp ra `/content/drive/MyDrive/CheXNet/train_base.log`.


In [ ]:
%cd /content/CheXNet
!python3 -u main.py \
  --mode train \
  --model-size base \
  --img-size 384 \
  --batch-size 48 \
  --epochs 14 \
  --data-dir /content/CheXNet/Database \
  --train-csv /content/CheXNet/Dataset/train_list.csv \
  --val-csv /content/CheXNet/Dataset/val_list.csv \
  --model-path /content/drive/MyDrive/CheXNet/hybrid_base.pth \
  --preload \
  --num-workers 8 2>&1 | tee /content/drive/MyDrive/CheXNet/train_base.log


---
## 🔥 Bước 3A: Huấn luyện Lại Bản `Large` Từ Đầu Với Code Mới (`ConvNeXtV2-Large + SwinV2-Large`, 384×384)

Để đảm bảo **so sánh công bằng tuyệt đối (`Fair Apples-to-Apples Comparison`)** với bản `Base` và phát huy tối đa hiệu quả của `Stochastic Depth (0.3)`, `5× Multi-Sample Dropout`, `Class-Conditional Hinge Sparsity + MIL Peak Loss` và `Adaptive ASL` ngay từ Epoch 1:
- Huấn luyện mới bản **`Large` (`279.12M` params)** trong **14 epochs**.
- Lưu ra file mới `/content/drive/MyDrive/CheXNet/hybrid_large_v2.pth` (giữ nguyên file `/content/drive/MyDrive/CheXNet/hybrid_model.pth` cũ làm mốc baseline `0.8759`).


In [ ]:
%cd /content/CheXNet
!python3 -u main.py \
  --mode train \
  --model-size large \
  --img-size 384 \
  --batch-size 32 \
  --epochs 14 \
  --data-dir /content/CheXNet/Database \
  --train-csv /content/CheXNet/Dataset/train_list.csv \
  --val-csv /content/CheXNet/Dataset/val_list.csv \
  --model-path /content/drive/MyDrive/CheXNet/hybrid_large_v2.pth \
  --preload \
  --num-workers 8 2>&1 | tee /content/drive/MyDrive/CheXNet/train_large_v2.log


---
## ⚡ Bước 3B (Tùy chọn Nhanh): Tinh chỉnh SOTA v2.1 Bản `Large` (5 Epochs từ Checkpoint Đỉnh Epoch 13)

Nếu bạn muốn có ngay một bản `Large` nâng cấp trong **~3.5 giờ** (trước khi chạy full 14 epochs ở Bước 3A), chạy ô dưới đây để tinh chỉnh từ `/content/drive/MyDrive/CheXNet/hybrid_model.pth` bằng chiến lược 3 pha SOTA v2.1.


In [ ]:
%cd /content/CheXNet
!python3 -u Finetune/main_finetune.py \
  --checkpoint /content/drive/MyDrive/CheXNet/hybrid_model.pth \
  --output /content/drive/MyDrive/CheXNet/hybrid_large_sota_v21.pth \
  --data-dir /content/CheXNet/Database \
  --train-csv /content/CheXNet/Dataset/train_list.csv \
  --val-csv /content/CheXNet/Dataset/val_list.csv \
  --batch-size 32 \
  --epochs 5 \
  --freeze-epochs 1 \
  --swa-start 4 \
  --mixup-alpha 0.2 \
  --rdrop-alpha 0.0 \
  --label-smoothing 0.02 \
  --preload \
  --num-workers 8 \
  --yes 2>&1 | tee /content/drive/MyDrive/CheXNet/finetune_large_v21.log


---
## 🏆 Bước 4: Đánh giá Test Set (4-View Multi-Resolution TTA) & So sánh Đối đầu `Base` vs. `Large` vs. `Ensemble`

Chạy `compare_models.py` trên toàn bộ tập Test (`12,427` ảnh) với **4-View Multi-Resolution TTA**:
- Tự động chọn `/content/drive/MyDrive/CheXNet/hybrid_large_v2.pth` (nếu đã train xong Bước 3A) hoặc `/content/drive/MyDrive/CheXNet/hybrid_large_sota_v21.pth` / `hybrid_model.pth` để đối sánh với `/content/drive/MyDrive/CheXNet/hybrid_base.pth`.
- Xuất đầy đủ bảng `AUROC`, `PR-AUC`, `F1@Thr*`, khoảng tin cậy **Bootstrap 95% CI**, kiểm định `p-value` và biểu đồ so sánh giữa `Base`, `Large` và **`Ensemble (Base + Large)`**.


In [ ]:
%cd /content/CheXNet
import os
large_ckpt = '/content/drive/MyDrive/CheXNet/hybrid_large_v2.pth'
if not os.path.exists(large_ckpt):
    large_ckpt = '/content/drive/MyDrive/CheXNet/hybrid_large_sota_v21.pth' if os.path.exists('/content/drive/MyDrive/CheXNet/hybrid_large_sota_v21.pth') else '/content/drive/MyDrive/CheXNet/hybrid_model.pth'
print(f'🔍 Sử dụng Large checkpoint: {large_ckpt}')

!python3 -u compare_models.py \
  --base-ckpt /content/drive/MyDrive/CheXNet/hybrid_base.pth \
  --large-ckpt {large_ckpt} \
  --data-dir /content/CheXNet/Database \
  --test-csv /content/CheXNet/Dataset/test_list.csv \
  --save-dir /content/drive/MyDrive/CheXNet/Results_Comparison \
  --batch-size 16 \
  --tta-views 4 2>&1 | tee /content/drive/MyDrive/CheXNet/compare_models.log
